In [2]:
using Revise
using MSUC
using Random
using Plots
using Gurobi
using Statistics
const GRB_ENV = Gurobi.Env()
optimizer=() -> Gurobi.Optimizer(GRB_ENV)

Set parameter Username
Set parameter LicenseID to value 2658456
Academic license - for non-commercial use only - expires 2026-04-28


#11 (generic function with 1 method)

In [3]:
T = 24
graph=MSUC.LinearGraph(T+1);

In [ ]:
function subproblem_builder(subproblem::Model, node::Int)
    # State variables
    H=[200, 200]
    @variable(subproblem, 0 <= volume[i in 1:2] <= H[i], MSUC.State, initial_value = 200)
    # Control variables
    @variables(subproblem, begin
        thermal_generation >= 0
        hydro_generation >= 0
        hydro_spill >= 0
    end)
    # Random variables
    @variable(subproblem, inflow)
    Ω = [0.0, 50.0, 100.0]
    P = [1 / 3, 1 / 3, 1 / 3]
    MSUC.parameterize(subproblem, Ω, P) do ω
        return JuMP.fix(inflow, ω)
    end
    # Transition function and constraints
    @constraints(
        subproblem,
        begin
            [i in 1:2], volume[i].out == volume[i].in - hydro_generation - hydro_spill + inflow
            demand_constraint, hydro_generation + thermal_generation == 150
        end
    )
    # Stage-objective
    fuel_cost = [50, 100, 150]
    @stageobjective(subproblem, fuel_cost[node] * thermal_generation)
    return subproblem
end

In [ ]:
model = MSUC.PolicyGraph(
    subproblem_builder,
    graph;
    sense = :Min,
    lower_bound = 0.0,
    optimizer = optimizer,
)

In [ ]:
MSUC.train(model; iteration_limit = 10)

In [5]:
instance=MSUC.parse_nc4("Data/T-Ramp/10_0_1_w.nc4",  optimizer, 24);1

1

In [ ]:
resSP=MSUC.benders_callback(instance, MSUC.extended_BD_FH_OH, silent=false, force=0.0, S=1, batch=1, gap=0.1, timelimit=120);

In [70]:
function subproblem_builder_UC(instance::MSUC.Instance, force::Float64, subproblem::Model, node::Int)
    # State variables
    N=instance.N
    thermal_units=values(instance.Thermalunits)
    Next=instance.Next
    Buses=1:size(Next)[1]
    Lines=values(instance.Lines)
    BusWind=instance.BusWind
    NumWindfarms=length(BusWind)
    @variable(subproblem, 0.0 <= power[i = 1:N] <= thermal_units[i].MaxPower, MSUC.State, initial_value = thermal_units[i].InitialPower)
    
    @variable(subproblem, 0.0 <= is_on[i = 1:N, t in 0:T] <= 1.0, MSUC.State, initial_value = 0.0) #initial value fictive

    # @variable(subproblem, is_on[i = 1:N, t in 0:T], Bin, MSUC.State, initial_value = 0.0) #initial value fictive
    @variable(subproblem, 0.0 <= start_up[i = 1:N, t in 1:T] <= 1.0, MSUC.State, initial_value = 0.0)
    @variable(subproblem, 0.0 <= start_down[i = 1:N, t in 1:T] <= 1.0, MSUC.State, initial_value = 0.0)

    # Control variables
    @variables(subproblem, begin
        power_shedding[b in Buses] >= 0
        power_curtailement[b in Buses] >= 0
        θ[b in Buses] 
        flow[b in Buses, bp in Next[b]]
    end)

    @variable(subproblem, gamma[i in 1:N, (a,b) in instance.Thermalunits[i].intervals]>=0)

    MSUC.thermal_unit_commit_constraints_SDDP(subproblem, instance)

    t=node

    # Random variables
    @variable(subproblem, error_forecast[k in 1:NumWindfarms])
    Ω = [[0.0 for b in BusWind]]
    P = [1.0]
    MSUC.parameterize(subproblem, Ω, P) do ω
        for k in 1:NumWindfarms
            JuMP.fix(error_forecast[k], ω[k])
        end
    end
    
    @constraint(subproblem, [i = 1: N], power[i].out <= sum(min(thermal_units[i].MaxPower, thermal_units[i].MinPower + thermal_units[i].DeltaRampDown * (b-t-1)) * gamma[i, [a,b]] for (a,b) in thermal_units[i].intervals_t[t] if b <= 24) + sum(thermal_units[i].MaxPower * gamma[i, [a,b]] for (a,b) in thermal_units[i].intervals_t[t] if b > 24))


    # Transition function and constraints
    @constraints(
        subproblem,
        begin
            [i = 1: N], power[i].out <= thermal_units[i].MaxPower * is_on[i, t].out
            [i = 1: N], power[i].out >= thermal_units[i].MinPower * is_on[i, t].out
            [i = 1: N], power[i].out - power[i].in <= (-thermal_units[i].DeltaRampUp)*start_up[i].out + (thermal_units[i].MinPower + thermal_units[i].DeltaRampUp)*is_on[i, t].out - (thermal_units[i].MinPower)*is_on[i, t-1].out
            [i = 1: N], power[i].in - power[i].out <= (-thermal_units[i].DeltaRampDown)*start_down[i].out + (thermal_units[i].MinPower + thermal_units[i].DeltaRampDown)*is_on[i, t-1].out - (thermal_units[i].MinPower)*is_on[i, t].out
            [line in Lines], flow[line.b1,line.b2] <= line.Fmax
            [line in Lines], flow[line.b1,line.b2] >= -line.Fmax
            [line in Lines], flow[line.b1,line.b2] == line.B12*(θ[line.b1]-θ[line.b2])
            [b in Buses], sum(power[unit.name].out for unit in thermal_units if unit.Bus==b) - power_curtailement[b] + power_shedding[b] == instance.Demandbus[b][node]*(1+force*sum(error_forecast[w] for w in 1:NumWindfarms if BusWind[w]==b))+sum(flow[b,bp] for bp in Next[b])
        end
    )
    # Stage-objective
    @stageobjective(subproblem, sum(unit.LinearTerm*power[unit.name].out for unit in thermal_units)+sum(MSUC.SHEDDING_COST*power_shedding[b]+MSUC.CURTAILEMENT_COST*power_curtailement[b] for b in Buses))
    return subproblem
end

subproblem_builder_UC (generic function with 1 method)

In [61]:
function masterproblem_builder_UC(instance::MSUC.Instance, masterproblem::Model)
    # State variables
    T= instance.TimeHorizon
    N=instance.N
    thermal_units=instance.Thermalunits

    @variable(masterproblem, is_on[i in 1:N, t in 0:T], Bin)
    @variable(masterproblem, start_up[i in 1:N, t in 1:T], Bin)
    @variable(masterproblem, start_down[i in 1:N, t in 1:T], Bin)

    @variable(masterproblem, power[i = 1:N])
    @constraint(masterproblem, [i = 1: N], power[i] == thermal_units[i].InitialPower)

    @variable(masterproblem, gamma[i in 1:N, (a,b) in instance.Thermalunits[i].intervals]>=0)
    @constraint(masterproblem, [unit in thermal_units, t in 1:T], sum(gamma[unit.name, [a,b]] for (a,b) in unit.intervals if a<=t<b)==is_on[unit.name,t])
    @constraint(masterproblem, [unit in thermal_units, t in 1:T], sum(gamma[unit.name, [a,b]] for (a,b) in unit.intervals if a==t)==start_up[unit.name,t])
    @constraint(masterproblem, [unit in thermal_units, t in 1:T], sum(gamma[unit.name, [a,b]] for (a,b) in unit.intervals if b==t)==start_down[unit.name,t])

    @constraint(masterproblem,  [unit in thermal_units, t in 1:T], is_on[unit.name, t]-is_on[unit.name, t-1]==start_up[unit.name, t]-start_down[unit.name, t])
    @constraint(masterproblem,  [unit in thermal_units, t in 1:T], start_up[unit.name, t]<=is_on[unit.name, t])
    @constraint(masterproblem,  [unit in thermal_units, t in 1:T], start_down[unit.name, t]<=1-is_on[unit.name, t])
    @constraint(masterproblem,  [unit in thermal_units, t in 1:T; unit.InitUpDownTime!=nothing], sum(start_up[unit.name, τ] for τ in max(1, t-unit.MinUpTime+1):t)+1*(t<unit.MinUpTime-unit.InitUpDownTime+1)*(unit.InitUpDownTime>0) <= is_on[unit.name, t])
    @constraint(masterproblem,  [unit in thermal_units, t in 1:T; unit.InitUpDownTime!=nothing], sum(start_down[unit.name, τ] for τ in max(1, t-unit.MinDownTime+1):t)+1*(t<unit.MinDownTime+unit.InitUpDownTime+1)*(unit.InitUpDownTime<0) <= 1-is_on[unit.name, t])
    @constraint(masterproblem,  [unit in thermal_units, t in 1:T; unit.InitUpDownTime==nothing], sum(start_up[unit.name, τ] for τ in max(1, t-unit.MinUpTime+1):t) <= is_on[unit.name, t])
    @constraint(masterproblem,  [unit in thermal_units, t in 1:T; unit.InitUpDownTime==nothing], sum(start_down[unit.name, τ] for τ in max(1, t-unit.MinDownTime+1):t) <= 1-is_on[unit.name, t])

    @constraint(masterproblem,  [unit in thermal_units; unit.InitUpDownTime!=nothing], is_on[unit.name, 0]==(unit.InitUpDownTime>=0))

    @constraint(masterproblem,  [unit in thermal_units; unit.InitUpDownTime==nothing], is_on[unit.name, 0]==0)

    for unit in thermal_units
        if (unit.InitialPower-unit.MinPower)/unit.DeltaRampDown <0
            limit=-1
        else
            limit=Int64(ceil((unit.InitialPower-unit.MinPower)/unit.DeltaRampDown))
        end
        for t in 0:limit
            @constraint(masterproblem, is_on[unit.name, t]==1)
        end
    end

    @stageobjective(masterproblem, sum(unit.ConstTerm*is_on[unit.name, t]+unit.StartUpCost*start_up[unit.name, t]+unit.StartDownCost*start_down[unit.name, t] for unit in thermal_units for t in 1:T))

    return masterproblem
end

masterproblem_builder_UC (generic function with 1 method)

In [ ]:
# force = 1.0
# model = MSUC.PolicyGraph(
#     subproblem_builder_UC,
#     masterproblem_builder_UC,
#     instance,
#     force,
#     graph;
#     sense = :Min,
#     lower_bound = 0.0,
#     optimizer = optimizer,
# )

A policy graph with 25 nodes.
 Node indices: 0, ..., 24


In [7]:
force = 1.0
model = MSUC.PolicyGraph(
    MSUC.subproblem_builder_UC_def,
    instance,
    force,
    graph;
    sense = :Min,
    lower_bound = 0.0,
    optimizer = optimizer,
)

MethodError: MethodError: no method matching zero(::Type{MSUC.State{VariableRef}})
The function `zero` exists, but no method is defined for this combination of argument types.

Closest candidates are:
  zero(!Matched::Type{Union{}}, Any...)
   @ Base number.jl:310
  zero(!Matched::Type{Dates.Date})
   @ Dates ~/julia-1.11.5/share/julia/stdlib/v1.11/Dates/src/types.jl:459
  zero(!Matched::Type{MutableArithmetics.Zero})
   @ MutableArithmetics ~/.julia/packages/MutableArithmetics/VMS0x/src/rewrite.jl:35
  ...


In [72]:
MSUC.train(model; iteration_limit = 2)

-------------------------------------------------------------------
         MSUC.jl (c) Oscar Dowson and contributors, 2017-25
-------------------------------------------------------------------
problem
  nodes           : 25
  state variables : 740
  scenarios       : 0.00000e+00
  existing cuts   : false
options
  solver          : serial mode
  risk measure    : MSUC.Expectation()
  sampling scheme : MSUC.InSampleMonteCarlo
subproblem structure
  VariableRef                             : [2709, 3453]
  AffExpr in MOI.EqualTo{Float64}         : [995, 1451]
  AffExpr in MOI.GreaterThan{Float64}     : [10, 10]
  AffExpr in MOI.LessThan{Float64}        : [40, 960]
  VariableRef in MOI.EqualTo{Float64}     : [1, 1]
  VariableRef in MOI.GreaterThan{Float64} : [1969, 2711]
  VariableRef in MOI.LessThan{Float64}    : [740, 741]
  VariableRef in MOI.ZeroOne              : [730, 730]
numerical stability report
  matrix range     [1e+00, 1e+03]
  objective range  [1e+00, 3e+03]
  bounds range

In [74]:
using JuMP
using Gurobi

# --- Création du modèle ---
model = Model(Gurobi.Optimizer)

# --- Variables ---
@variable(model, x >= 0)          # quantité du produit A
@variable(model, y >= 0)          # quantité du produit B
@variable(model, z >= 0)          # 1 si la machine est allumée
JuMP.set_binary(z)                # rendre z binaire après coup
# alternative explicite : replacez la déclaration par `@variable(model, z, Bin)` si vous pouvez redéfinir la variable

# --- Données ---
profit_A = 3.0
profit_B = 5.0
hours_A  = 1.0
hours_B  = 2.0
capacity = 6.0
bigM     = 6.0   # permet de lier la contrainte à la variable binaire

JuMP.fix(z, 0.0; force=true)

# --- Contraintes ---
cstr = @constraint(model, hours_A * x + hours_B * y <= capacity * z)
@constraint(model, x <= 4)    # limite sur la production de A
@constraint(model, y <= 3)    # limite sur la production de B

# --- Objectif ---
@objective(model, Max, profit_A * x + profit_B * y - 2 * z)  # -2 = coût d’activation

# --- Résolution ---
optimize!(model)

# --- Résultats ---
println("x = ", value(x))
println("y = ", value(y))
println("z = ", value(z))
println("Profit = ", objective_value(model))
println(JuMP.dual(JuMP.FixRef(z)))
println(dual(cstr))


Set parameter Username
Set parameter LicenseID to value 2658456
Academic license - for non-commercial use only - expires 2026-04-28
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (linux64 - "Ubuntu 22.04.5 LTS")

CPU model: Intel(R) Core(TM) Ultra 9 185H, instruction set [SSE2|AVX|AVX2]
Thread count: 22 physical cores, 22 logical processors, using up to 22 threads

Optimize a model with 3 rows, 3 columns and 5 nonzeros
Model fingerprint: 0x245ece91
Variable types: 2 continuous, 1 integer (1 binary)
Coefficient statistics:
  Matrix range     [1e+00, 6e+00]
  Objective range  [2e+00, 5e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [3e+00, 4e+00]
Found heuristic solution: objective -0.0000000
Presolve removed 3 rows and 3 columns
Presolve time: 0.00s
Presolve: All rows and columns removed

Explored 0 nodes (0 simplex iterations) in 0.00 seconds (0.00 work units)
Thread count was 1 (of 22 available processors)

Solution count 1: -0 
No other solutions better than -0

Optimal 

ErrorException: Gurobi Error 10005: Unable to retrieve attribute 'RC'

In [ ]:
println(model[1].subproblem)

In [ ]:
unit = instance.Thermalunits[10]
unit.DeltaRampDown

In [ ]:
println(model[1].subproblem)

In [ ]:
model[1].bellman_function.global_theta.cuts

In [ ]:
for i in 1:10
    println((instance.Thermalunits[i].InitialPower, instance.Thermalunits[i].MinPower, instance.Thermalunits[i].MaxPower, instance.Thermalunits[i].DeltaRampDown, instance.Thermalunits[i].DeltaRampUp))
end